# 2D Trajectory Visualization: Drift + Diffusion

Visualizes the Langevin dynamics that memories undergo in the 2D guided-drift sandbox.
Each step applies:

$$\mu_{t+1} = \mu_t + \sigma \cdot \varepsilon_t + \eta \cdot \text{score}(\mu_t)$$

where $\varepsilon_t \sim \mathcal{N}(0, I)$, $\sigma$ is diffusive noise, and $\eta$ is the drift step size.

We compare **unit-normalized** score ($\hat{s} = s / \|s\|$) vs **raw** score ($\nabla_x \log p(x)$).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys, os

# Ensure project root is on path
sys.path.insert(0, os.path.abspath('..'))

from src.model.analytic_gmm_2d import AnalyticGMM2D
from utls.sandbox_2d_data import make_2d_grid_stimuli
from utls.analysis_2d import plot_prior_with_score_field

# ── Setup ──
gmm = AnalyticGMM2D()
X0, name_to_idx, stimulus_pool = make_2d_grid_stimuli()
X0 = X0.to(torch.float64)

# Match runner's noise scaling (utls/runners_2d.py:75-77)
dim_std = X0.std(0, unbiased=True)
rms_std = torch.sqrt(torch.mean(dim_std ** 2)).item()
scaled_std = dim_std / rms_std

print(f'Stimuli: {X0.shape[0]} points, dim_std={dim_std.numpy()}, rms_std={rms_std:.4f}')

In [ ]:
def simulate_trajectories(
    gmm, X0, indices, *,
    sigma=0.3, sigma0=0.5, drift_step_size=0.05,
    n_steps=100, normalize_score=True, seed=42,
):
    """Simulate memory trajectories with drift + diffusion.

    Matches the runner update loop (utls/runners_2d.py:96-119):
      1. Diffusion: mu += sigma * scaled_std * eps
      2. Drift:     mu += drift_step_size * score(mu)

    Parameters
    ----------
    normalize_score : bool
        True = unit-norm score (what the runner uses via ScoreAdapter2D).
        False = raw score (gradient magnitude varies with position).

    Returns
    -------
    trajectories : list of np.ndarray, each shape [n_steps+1, 2]
    """
    rng = torch.Generator().manual_seed(seed)
    trajectories = []

    for idx in indices:
        x_clean = X0[idx].clone().to(torch.float64)
        # Encoding noise (runner line 142-147)
        enc_noise = torch.randn(x_clean.shape, generator=rng, dtype=torch.float64)
        x = x_clean + enc_noise * (sigma0 * dim_std)
        traj = [x.numpy().copy()]

        for _ in range(n_steps):
            # 1. Diffusion (runner lines 98-107)
            eps = torch.randn(x.shape, generator=rng, dtype=torch.float64)
            x = x + eps * (sigma * scaled_std)

            # 2. Drift (runner lines 110-119)
            if drift_step_size > 0:
                s = gmm.score(x)
                if normalize_score:
                    s = s / (s.norm() + 1e-8)
                x = x + drift_step_size * s

            traj.append(x.numpy().copy())

        trajectories.append(np.array(traj))

    return trajectories

In [ ]:
def plot_trajectories(ax, gmm, trajectories, title, colors, grid_n=40):
    """Plot GMM density contours with trajectory overlays."""
    # Density contours
    xs = np.linspace(-5, 5, grid_n)
    ys = np.linspace(-5, 5, grid_n)
    Xg, Yg = np.meshgrid(xs, ys)
    pts = torch.tensor(np.stack([Xg.ravel(), Yg.ravel()], axis=1), dtype=torch.float64)
    log_p = gmm.log_prob(pts).numpy().reshape(grid_n, grid_n)
    ax.contourf(Xg, Yg, np.exp(log_p), levels=20, cmap='Blues', alpha=0.6)
    ax.contour(Xg, Yg, np.exp(log_p), levels=8, colors='steelblue', linewidths=0.5, alpha=0.4)

    # GMM means
    means = gmm.means.numpy()
    ax.scatter(means[:, 0], means[:, 1], marker='+', s=120, c='red', linewidths=2, zorder=20)

    # Trajectories
    for ci, traj in enumerate(trajectories):
        c = colors[ci]
        ax.plot(traj[:, 0], traj[:, 1], '-', color=c, lw=1.0, alpha=0.7)
        ax.scatter(traj[0, 0], traj[0, 1], s=50, color=c, edgecolors='k', linewidths=0.5, zorder=10)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=80, marker='*', color=c, edgecolors='k', linewidths=0.5, zorder=10)

    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')

## Normed vs Raw Score

**Left**: Unit-normalized score — constant drift step size regardless of position (what the runner uses).  
**Right**: Raw score — drift magnitude scales with $\|\nabla \log p(x)\|$, so pull is stronger far from modes.

In [ ]:
n_traj = 8
rng_sel = np.random.default_rng(77)
indices = rng_sel.choice(len(X0), size=n_traj, replace=False)
colors = plt.cm.Set1(np.linspace(0, 1, n_traj))

SIGMA = 0.3
SIGMA0 = 0.5
ETA = 0.05
N_STEPS = 100
SEED = 42

traj_normed = simulate_trajectories(
    gmm, X0, indices,
    sigma=SIGMA, sigma0=SIGMA0, drift_step_size=ETA,
    n_steps=N_STEPS, normalize_score=True, seed=SEED,
)
traj_raw = simulate_trajectories(
    gmm, X0, indices,
    sigma=SIGMA, sigma0=SIGMA0, drift_step_size=ETA,
    n_steps=N_STEPS, normalize_score=False, seed=SEED,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_trajectories(ax1, gmm, traj_normed, f'Unit-normalized score (σ={SIGMA}, η={ETA})', colors)
plot_trajectories(ax2, gmm, traj_raw, f'Raw score (σ={SIGMA}, η={ETA})', colors)
fig.suptitle('Drift + Diffusion Trajectories: Normed vs Raw Score', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Effect of Diffusion Magnitude

All panels use **unit-normalized** score (runner default). Increasing $\sigma$ adds more per-step noise,
making trajectories noisier and less deterministic.

In [ ]:
sigmas = [0.0, 0.3, 0.8]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, sig in zip(axes, sigmas):
    trajs = simulate_trajectories(
        gmm, X0, indices,
        sigma=sig, sigma0=SIGMA0, drift_step_size=ETA,
        n_steps=N_STEPS, normalize_score=True, seed=SEED,
    )
    label = 'drift only' if sig == 0 else f'σ={sig}'
    plot_trajectories(ax, gmm, trajs, f'Normed score, {label}', colors)

fig.suptitle('Effect of Diffusion Magnitude (unit-normalized score)', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Long-Run Convergence to Prior Modes

With 500 steps, trajectories should converge toward GMM modes.
Raw score pulls harder from far away, so convergence behavior differs.

In [ ]:
N_LONG = 500

traj_long_normed = simulate_trajectories(
    gmm, X0, indices,
    sigma=SIGMA, sigma0=SIGMA0, drift_step_size=ETA,
    n_steps=N_LONG, normalize_score=True, seed=SEED,
)
traj_long_raw = simulate_trajectories(
    gmm, X0, indices,
    sigma=SIGMA, sigma0=SIGMA0, drift_step_size=ETA,
    n_steps=N_LONG, normalize_score=False, seed=SEED,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_trajectories(ax1, gmm, traj_long_normed, f'Normed, {N_LONG} steps', colors)
plot_trajectories(ax2, gmm, traj_long_raw, f'Raw, {N_LONG} steps', colors)
fig.suptitle('Long-Run Convergence: Normed vs Raw Score', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()